## pcFVA Dataframes for Plots for G6PD

Necessary imports

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests
from rbc_gem_utils import read_cobra_model
from rbc_gem_utils.analysis.overlay import load_overlay_model

### A few file paths

In [2]:
model_id = "RBC3P_expanded"
dataset_name = "RBComics_G6PD"
data_path = Path("../../../../data/analysis/OVERLAY").resolve()
root_path = Path("../../../..").resolve()
results_path = root_path / "data" / "processed" / model_id / "OVERLAY"
pcmodel_dirpath = data_path / model_id
dataset_path = results_path / dataset_name
dataset_models_dirpath = dataset_path / "pcmodels"
pcfva_results_dirpath = dataset_path / "pcFVA"
corr_results_dirpath = dataset_path / "correlations"
df_pcfva_all_filename = pcfva_results_dirpath / f"{model_id}_PC_FVAresults_ALL.tsv"
model_filename = pcmodel_dirpath / f"{model_id}.xml"
pcmodel_filename = pcmodel_dirpath / f"{model_id}_PC.xml"
correlations2_dirpath = dataset_path / "correlations2"

print(results_path)
print(pcmodel_dirpath)
print(dataset_path)
print(dataset_models_dirpath)
print(pcfva_results_dirpath)
print(corr_results_dirpath)
print(df_pcfva_all_filename)
print(model_filename)
print(pcmodel_filename)
print(correlations2_dirpath)

D:\Projects\RBC-GEM-akey7\data\processed\RBC3P_expanded\OVERLAY
D:\Projects\RBC-GEM-akey7\data\analysis\OVERLAY\RBC3P_expanded
D:\Projects\RBC-GEM-akey7\data\processed\RBC3P_expanded\OVERLAY\RBComics_G6PD
D:\Projects\RBC-GEM-akey7\data\processed\RBC3P_expanded\OVERLAY\RBComics_G6PD\pcmodels
D:\Projects\RBC-GEM-akey7\data\processed\RBC3P_expanded\OVERLAY\RBComics_G6PD\pcFVA
D:\Projects\RBC-GEM-akey7\data\processed\RBC3P_expanded\OVERLAY\RBComics_G6PD\correlations
D:\Projects\RBC-GEM-akey7\data\processed\RBC3P_expanded\OVERLAY\RBComics_G6PD\pcFVA\RBC3P_expanded_PC_FVAresults_ALL.tsv
D:\Projects\RBC-GEM-akey7\data\analysis\OVERLAY\RBC3P_expanded\RBC3P_expanded.xml
D:\Projects\RBC-GEM-akey7\data\analysis\OVERLAY\RBC3P_expanded\RBC3P_expanded_PC.xml
D:\Projects\RBC-GEM-akey7\data\processed\RBC3P_expanded\OVERLAY\RBComics_G6PD\correlations2


### Load main models

In [3]:
model = read_cobra_model(filename=model_filename)
pcmodel = load_overlay_model(filename=pcmodel_filename)

Set parameter Username


### Load all results from pcFVA

In [4]:
df_pcfva_all = pd.read_csv(df_pcfva_all_filename, sep="\t")
df_pcfva_all

,reactions,optimum,model,minimum,maximum
0,ADA,0.00,RBC3P_expanded_PC_Mean_D10,0.0,0.278274
1,ADA,0.00,RBC3P_expanded_PC_Mean_D23,0.0,0.282364
2,ADA,0.00,RBC3P_expanded_PC_Mean_D42,0.0,0.277745
3,ADA,0.00,RBC3P_expanded_PC_Median_D10,0.0,0.273331
4,ADA,0.00,RBC3P_expanded_PC_Median_D23,0.0,0.274004
...,...,...,...,...,...
7922875,TPI,0.99,RBC3P_expanded_PC_S00649_D23,0.0,0.001753
7922876,TPI,0.99,RBC3P_expanded_PC_S00649_D42,0.0,0.001790
7922877,TPI,0.99,RBC3P_expanded_PC_S00650_D10,0.0,0.001541
7922878,TPI,0.99,RBC3P_expanded_PC_S00650_D23,0.0,0.001325


### Separate reaction dataframe by reaction type

In [5]:
# Initialize entries with prefixes used for seperating DataFrames
dict_of_dataframes_types = {
    "reactions": None,
    "proteins": "PROTDL",
    # "complexes": "CPLXFM",
    # "complex_dilutions": "CPLXDL",
    "enzymes": "ENZDL",
    # "enzyme_formation": "ENZFM",
    "budgets": "PBDL",
    "relaxation": "RELAX",
}
for key, prefix in dict_of_dataframes_types.copy().items():
    if prefix:
        df = df_pcfva_all[
            df_pcfva_all["reactions"].apply(lambda x: x.startswith(prefix))
        ]
    else:
        df = df_pcfva_all[
            df_pcfva_all["reactions"].apply(lambda x: x in model.reactions)
        ]
    dict_of_dataframes_types[key] = df.copy()

### Always abundance independent reactions

In [6]:
groupby_list = ["model", "reactions"]
always_abundance_independent = [
    r.id for r in model.reactions.query(lambda x: not x.boundary and not x.genes)
]
print(
    f"Number of reactions w/o genes, always expression independent: {len(always_abundance_independent)}"
)
always_abundance_independent

Number of reactions w/o genes, always expression independent: 7


['H2O2t', 'Ht', 'PIt', 'NTPA', 'FE2O2OX', 'FE2H2O2X', 'HOXG']

### Get maximum flux range

In [7]:
df = dict_of_dataframes_types["reactions"].copy()
df = df.groupby(groupby_list)[["minimum", "maximum"]].agg(
    {
        "minimum": "min",
        "maximum": "max",
    }
)
df_max_flux_per_model = df.abs().max(axis=1)
df_max_flux_per_model.name = "Flux"
df_max_flux_per_model

model                         reactions
RBC3P_expanded_PC_Mean_D10    ADA          0.278274
                              ADEt         0.018362
                              ADK1         0.018362
                              ADNK1        0.018362
                              ADNt         0.278274
                                             ...   
RBC3P_expanded_PC_S00650_D42  SPODM        0.171777
                              TALA         0.041669
                              TKT1         0.041669
                              TKT2         0.041669
                              TPI          0.045783
Name: Flux, Length: 177898, dtype: float64

### Get range of maximum flux

In [8]:
df = dict_of_dataframes_types["reactions"].copy()
df["Range"] = df["maximum"] - df["minimum"]
df_flux_range_per_model = df.groupby(groupby_list)["Range"].max()
df_flux_range_per_model

model                         reactions
RBC3P_expanded_PC_Mean_D10    ADA          0.278274
                              ADEt         0.018362
                              ADK1         0.018362
                              ADNK1        0.018362
                              ADNt         0.296636
                                             ...   
RBC3P_expanded_PC_S00650_D42  SPODM        0.171777
                              TALA         0.082190
                              TKT1         0.082190
                              TKT2         0.082190
                              TPI          0.045783
Name: Range, Length: 177898, dtype: float64

### Get maximum abundance

In [9]:
min_reaction_list = model.reactions.query(lambda x: x.gene_reaction_rule).list_attr(
    "id"
)
reaction_enzymes_map = {
    rid: tuple(
        pcmodel.reactions.query(
            lambda x: x.id.startswith(f"ENZDL_enzyme_{rid}_")
        ).list_attr("id")
    )
    for rid in min_reaction_list
}
enzyme_reaction_map = {
    enzyme: rid for rid, enzymes in reaction_enzymes_map.items() for enzyme in enzymes
}
df = dict_of_dataframes_types["enzymes"].copy()
df["reactions"] = df["reactions"].apply(lambda x: enzyme_reaction_map[x])
df_max_enzyme_per_model = df.groupby(groupby_list)["maximum"].max()
df_max_enzyme_per_model.name = "Abundance"
df_max_enzyme_per_model

model                         reactions
RBC3P_expanded_PC_Mean_D10    ADA           2.089841
                              ADEt          4.387130
                              ADK1         22.533588
                              ADNK1         2.487410
                              ADNt          3.092031
                                             ...    
RBC3P_expanded_PC_S00650_D42  SPODM        28.826502
                              TALA          2.055036
                              TKT1          0.847117
                              TKT2          0.847117
                              TPI          12.508601
Name: Abundance, Length: 111874, dtype: float64

### Merge results above into one DataFrame

In [10]:
df_reaction_flux_abundance = (
    pd.merge(
        df_max_flux_per_model,
        df_flux_range_per_model,
        left_index=True,
        right_index=True,
    )
    .merge(df_max_enzyme_per_model, left_index=True, right_index=True)
    .reset_index(drop=False)
)
df_reaction_flux_abundance

,model,reactions,Flux,Range,Abundance
0,RBC3P_expanded_PC_Mean_D10,ADA,0.278274,0.278274,2.089841
1,RBC3P_expanded_PC_Mean_D10,ADEt,0.018362,0.018362,4.387130
2,RBC3P_expanded_PC_Mean_D10,ADK1,0.018362,0.018362,22.533588
3,RBC3P_expanded_PC_Mean_D10,ADNK1,0.018362,0.018362,2.487410
4,RBC3P_expanded_PC_Mean_D10,ADNt,0.278274,0.296636,3.092031
...,...,...,...,...,...
111869,RBC3P_expanded_PC_S00650_D42,SPODM,0.171777,0.171777,28.826502
111870,RBC3P_expanded_PC_S00650_D42,TALA,0.041669,0.082190,2.055036
111871,RBC3P_expanded_PC_S00650_D42,TKT1,0.041669,0.082190,0.847117
111872,RBC3P_expanded_PC_S00650_D42,TKT2,0.041669,0.082190,0.847117


## Calculate correlations between flux and abundance

Don't include mean/median models in correlation calculations.

In [11]:
df_flux_abundance_correlation_step_1 = df_reaction_flux_abundance.loc[
    :, ["model", "reactions", "Flux", "Abundance"]
]
filter_mask = df_flux_abundance_correlation_step_1["model"].apply(
    lambda x: not ("Mean" in x or "Median" in x)
)
df_flux_abundance_correlation_step_1 = df_flux_abundance_correlation_step_1[filter_mask]
df_flux_abundance_correlation_step_1

,model,reactions,Flux,Abundance
366,RBC3P_expanded_PC_S00001_D10,ADA,0.279580,2.079029
367,RBC3P_expanded_PC_S00001_D10,ADEt,0.017500,3.958645
368,RBC3P_expanded_PC_S00001_D10,ADK1,0.017500,33.391864
369,RBC3P_expanded_PC_S00001_D10,ADNK1,0.017500,2.477086
370,RBC3P_expanded_PC_S00001_D10,ADNt,0.272022,2.484688
...,...,...,...,...
111869,RBC3P_expanded_PC_S00650_D42,SPODM,0.171777,28.826502
111870,RBC3P_expanded_PC_S00650_D42,TALA,0.041669,2.055036
111871,RBC3P_expanded_PC_S00650_D42,TKT1,0.041669,0.847117
111872,RBC3P_expanded_PC_S00650_D42,TKT2,0.041669,0.847117


In [12]:
def calc_raw_p_values(group_df):
    fluxes = group_df["Flux"]
    abundances = group_df["Abundance"]
    rho, raw_p_value = spearmanr(fluxes, abundances)
    raw_p_value = (
        1.0 if np.isnan(raw_p_value) else raw_p_value
    )  # To enable p-value correction, make missing p-values as insignificant as possible
    return pd.DataFrame([{"rho": rho, "raw_p_value": raw_p_value}])


df_flux_abundance_correlation = (
    df_flux_abundance_correlation_step_1.drop("model", axis=1)
    .groupby("reactions")
    .apply(calc_raw_p_values, include_groups=False)
    .reset_index(level=1, drop=True)
)
df_flux_abundance_correlation

C:\Users\Alicia Key\AppData\Local\Temp\ipykernel_24008\694070394.py:4: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho, raw_p_value = spearmanr(fluxes, abundances)


,rho,raw_p_value
reactions,,
ADA,-0.069060,3.135064e-03
ADEt,-0.078595,7.702784e-04
ADK1,-0.096827,3.372097e-05
ADNK1,0.238054,5.734865e-25
ADNt,0.511792,1.361003e-122
...,...,...
SPODM,0.058384,1.253744e-02
TALA,0.083149,3.725993e-04
TKT1,0.208445,2.155061e-19


### Correct p-values

In [13]:
_, pvals_corrected, _, _ = multipletests(
    df_flux_abundance_correlation["raw_p_value"], alpha=0.05, method="fdr_bh"
)
neg_log10_adj_p_values = -np.log10(pvals_corrected)
neg_log10_pval_max = max(neg_log10_adj_p_values[~np.isinf(-np.log10(pvals_corrected))])
neg_log10_adj_p_values[np.isinf(neg_log10_adj_p_values)] = neg_log10_pval_max
df_flux_abundance_correlation["adj_p_value"] = pvals_corrected
df_flux_abundance_correlation["neg_log10_adj_p_value"] = neg_log10_adj_p_values

# Replace NaN correlations with 0.0
df_flux_abundance_correlation["rho"] = df_flux_abundance_correlation["rho"].apply(
    lambda x: 0.0 if np.isnan(x) else x
)


def categorize_abundance_dependence(rho):
    if rho >= 0.8:
        return "Dependent"
    elif 0.5 <= rho < 0.8:
        return "Correlated"
    else:
        return "Independent"


df_flux_abundance_correlation["abundance_dependence"] = df_flux_abundance_correlation[
    "rho"
].apply(lambda rho: categorize_abundance_dependence(rho))

df_flux_abundance_correlation

C:\Users\Alicia Key\AppData\Local\Temp\ipykernel_24008\4036705117.py:4: RuntimeWarning: divide by zero encountered in log10
  neg_log10_adj_p_values = -np.log10(pvals_corrected)
C:\Users\Alicia Key\AppData\Local\Temp\ipykernel_24008\4036705117.py:5: RuntimeWarning: divide by zero encountered in log10
  neg_log10_pval_max = max(neg_log10_adj_p_values[~np.isinf(-np.log10(pvals_corrected))])


,rho,raw_p_value,adj_p_value,neg_log10_adj_p_value,abundance_dependence
reactions,,,,,
ADA,-0.069060,3.135064e-03,4.068913e-03,2.390522,Independent
ADEt,-0.078595,7.702784e-04,1.021456e-03,2.990780,Independent
ADK1,-0.096827,3.372097e-05,4.783672e-05,4.320239,Independent
ADNK1,0.238054,5.734865e-25,1.249381e-24,23.903305,Independent
ADNt,0.511792,1.361003e-122,4.151060e-122,121.381841,Correlated
...,...,...,...,...,...
SPODM,0.058384,1.253744e-02,1.560783e-02,1.806657,Independent
TALA,0.083149,3.725993e-04,5.165582e-04,3.286881,Independent
TKT1,0.208445,2.155061e-19,4.108084e-19,18.386361,Independent


### For easier EDA, display a Plotly plot

In [14]:
fig = px.scatter(
    df_flux_abundance_correlation.reset_index(),
    x="rho",
    y="neg_log10_adj_p_value",
    hover_data=["reactions", "abundance_dependence"],
    title="Flux, Abundance Correlations",
)
fig.show()

### Look for nas

In [15]:
df_flux_abundance_correlation.isna().sum()

rho                      0
raw_p_value              0
adj_p_value              0
neg_log10_adj_p_value    0
abundance_dependence     0
dtype: int64

### Put categories in the rows for making donut charts

In [16]:
df_pathways_filename = root_path / "data" / "curation" / f"subsystems.tsv"
df_pathways = pd.read_csv(
    df_pathways_filename, sep="\t", index_col=0, dtype=str
)
df_pathways

,category,kegg.pathway.name,kegg.pathway,notes
name,,,,
"Alanine, aspartate and glutamate metabolism",Amino acid metabolism,"Alanine, aspartate and glutamate metabolism",hsa00250,NaN
Arginine and proline metabolism,Amino acid metabolism,Arginine and proline metabolism,hsa00330,NaN
Cysteine and methionine metabolism,Amino acid metabolism,Cysteine and methionine metabolism,hsa00270,NaN
"Glycine, serine and threonine metabolism",Amino acid metabolism,"Glycine, serine and threonine metabolism",hsa00260,NaN
Histidine metabolism,Amino acid metabolism,Histidine metabolism,hsa00340,NaN
...,...,...,...,...
Aminoacyl-tRNA biosynthesis,Translation,Aminoacyl-tRNA biosynthesis,has00970,NaN
"Transport, extracellular",Transport reactions,NaN,NaN,Representative subsystem for all transport rea...
5-fluorouracil metabolism,Xenobiotics biodegradation and metabolism,Drug metabolism - other enzymes,hsa00983,"Subnetwork of KEGG pathway ""Drug metabolism - ..."


In [17]:
df_reaction_subsystem_rows = []
for reaction in model.reactions:
    df_reaction_subsystem_rows.append(
        {"reactions": reaction.id, "subsystem": reaction.subsystem}
    )
df_reaction_subsystem = pd.DataFrame(df_reaction_subsystem_rows)
df_flux_abundance_correlation = df_flux_abundance_correlation.merge(
    df_reaction_subsystem, how="left", on="reactions"
)
df_patwhay_columns_to_merge = df_pathways.reset_index().loc[:, ["name", "category"]]
df_flux_abundance_correlation = df_flux_abundance_correlation.merge(df_patwhay_columns_to_merge, how="left", left_on="subsystem", right_on="name")
df_flux_abundance_correlation.drop("name", axis=1, inplace=True)
df_flux_abundance_correlation

,reactions,rho,raw_p_value,adj_p_value,neg_log10_adj_p_value,abundance_dependence,subsystem,category
0,ADA,-0.069060,3.135064e-03,4.068913e-03,2.390522,Independent,Purine metabolism,Nucleotide metabolism
1,ADEt,-0.078595,7.702784e-04,1.021456e-03,2.990780,Independent,"Transport, extracellular",Transport reactions
2,ADK1,-0.096827,3.372097e-05,4.783672e-05,4.320239,Independent,Purine metabolism,Nucleotide metabolism
3,ADNK1,0.238054,5.734865e-25,1.249381e-24,23.903305,Independent,Purine metabolism,Nucleotide metabolism
4,ADNt,0.511792,1.361003e-122,4.151060e-122,121.381841,Correlated,"Transport, extracellular",Transport reactions
...,...,...,...,...,...,...,...,...
56,SPODM,0.058384,1.253744e-02,1.560783e-02,1.806657,Independent,Reactive species formation and detoxification,Reactive species
57,TALA,0.083149,3.725993e-04,5.165582e-04,3.286881,Independent,Pentose phosphate pathway,Carbohydrate metabolism
58,TKT1,0.208445,2.155061e-19,4.108084e-19,18.386361,Independent,Pentose phosphate pathway,Carbohydrate metabolism
59,TKT2,0.208445,2.155061e-19,4.108084e-19,18.386361,Independent,Pentose phosphate pathway,Carbohydrate metabolism


### Save correlations DataFrame for plotting in another notebook

In [18]:
df_flux_abundance_correlation_filename = (
    correlations2_dirpath / "df_flux_abundance_correlation.csv"
)
df_flux_abundance_correlation.to_csv(df_flux_abundance_correlation_filename, index=True)

### Data for stacked bars of proteins vs number of reactions

TODO: Sort each set of genes before concatenation!

In [19]:
df_reaction_gene_rows = []
for reaction in model.reactions:
    genes = ";".join(gene.id for gene in reaction.genes)
    df_reaction_gene_rows.append({"reactions": reaction.id, "genes": genes})
df_reaction_gene = pd.DataFrame(df_reaction_gene_rows)
df_reaction_gene

,reactions,genes
0,ADA,ADA
1,ADEt,SLC29A1;SLC43A3
2,ADK1,AK1
3,ADNK1,ADK
4,ADNt,SLC29A1
...,...,...
92,DM_oh1_c,
93,DM_oh1r_c,
94,DM_fe2_c,
95,SK_pheme_c,


In [20]:
df_reaction_gene_abundance_dependence = df_reaction_gene.merge(
    df_flux_abundance_correlation, "left"
)[["reactions", "genes", "abundance_dependence"]]
df_reaction_gene_abundance_dependence

,reactions,genes,abundance_dependence
0,ADA,ADA,Independent
1,ADEt,SLC29A1;SLC43A3,Independent
2,ADK1,AK1,Independent
3,ADNK1,ADK,Independent
4,ADNt,SLC29A1,Correlated
...,...,...,...
92,DM_oh1_c,,NaN
93,DM_oh1r_c,,NaN
94,DM_fe2_c,,NaN
95,SK_pheme_c,,NaN


In [21]:
df_gene_reaction_count = (
    df_reaction_gene_abundance_dependence[df_reaction_gene_abundance_dependence["genes"] != ""]
    .groupby(["abundance_dependence", "genes"])
    .count()
    .sort_values("reactions", ascending=False)
)
df_gene_reaction_count

reactions
abundance_dependence genes                                                        
Independent          HBZ;HBA;HBG1;HBE1;HBB;HBM;HBG2;HBD;HBQ1                     4
Correlated           EMB;SLC16A1;SLC16A7;BSG                                     2
Dependent            NT5C2                                                       2
Correlated           SLC29A1                                                     2
Independent          TKT                                                         2
                     SLC29A1;SLC43A3                                             2
                     PGAM1;BPGM                                                  2
Dependent            AQP3;AQP1                                                   1
                     AMPD3                                                       1
                     AQP1                                                        1
                     ATP2B4;ATP2B1                                               1
                     PANX1;P2RX7;SCN4A;PIEZO1;GRIA1;P2RX1;SCN9A;P2RX...          1
                     PANX1;TRPC6;P2RX7;CACNA1A;TMEM109;PIEZO1;GRIA1;...          1
                     PFKL;PFKM;PFKP                                              1
                     PKLR;PKM                                                    1
                     RHD;RHAG;AQP1;AQP3;RHCE                                     1
                     G6PD                                                        1
                     GPI                                                         1
                     P2RX2;PANX1;SLC22A4;P2RX7;KCNN4;TMEM109;PIEZO1;...          1
Correlated           ENO1;ENO3;ENO2                                              1
Independent          ADK                                                         1
                     ADA                                                         1
                     ABCC4                                                       1
Dependent            RHD;RHAG;RHCE;AQP1                                          1
Independent          BLVRB;BLVRA                                                 1
                     BPGM                                                        1
                     AK1                                                         1
                     ALDOC;ALDOA;ALDOB                                           1
                     CYB5R3;CYB5B;CYB5A                                          1
                     CYB5B;HBZ;HBA;HBG1;HBE1;CYB5A;HBB;HBG2;HBM;HBD;...          1
                     GPX1;GPX4                                                   1
                     GAPDHS;GAPDH                                                1
                     GSR                                                         1
                     HK3;HK1                                                     1
                     APRT                                                        1
                     ATP1A3;ATP1A4;ATP1B3;ATP1B2;ATP1B1;ATP1A1                   1
                     LDHB;LDHA                                                   1
                     HPRT1                                                       1
                     PGD                                                         1
                     PGK2;PGK1                                                   1
                     PNP                                                         1
                     PPA2;LHPP;PPA1;PRUNE1                                       1
                     PGLS                                                        1
                     PGM2                                                        1
                     RPE                                                         1
                     PRPS1L1;PRPS1;PRPS2                                         1
                     SLC2A14;SLC2A1;SLC2A2;SLC2A3;SLC2A4                         1
      